# Plan and schema diversity metrics

Captures each workload's compiled plans with `EXPLAIN` against the live lakehouse (cached in
`workloads/<w>/plans.json`), then computes the per-workload plan and schema diversity metrics read by Tables 4
and 5 in `tables.ipynb`. Writes `results/analysis/workload_metrics.csv`. Needs a deployed lakehouse with the
TPC-DS schema loaded.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
for _p in (str(ROOT), str(Path.cwd())):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from config import WORKLOADS_DIR, TRINO_CATALOG, TRINO_SCHEMA
from lakehouse.lakehouse import Lakehouse
from analysis import analyse_workload_set, capture_query_plans_for_set
from workloads import ROWS, PLAN_METRICS_KEY, ANALYSIS_RESULTS_DIR, WORKLOAD_METRICS_CSV

INSTANCE_NAME = "lakehouse-h"   # Kubernetes release of the lakehouse
NAMESPACE = ""                  # Kubernetes namespace it is deployed in

WORKLOAD_NAMES = [PLAN_METRICS_KEY.get(w, w) for w in ROWS]

lh = Lakehouse.from_release(instance_name=INSTANCE_NAME, namespace=NAMESPACE, sync_schemas=True, verbose=False)

## 1 · Capture compiled plans


In [ ]:
plans_summary_df, plans_failures_df = capture_query_plans_for_set(
    WORKLOAD_NAMES,
    workload_root=WORKLOADS_DIR,
    lh=lh,
    catalog=TRINO_CATALOG,
    schema=TRINO_SCHEMA,
    force_recompute=False,
    plan_workers=8,
)
if not plans_failures_df.empty:
    display(plans_failures_df)
plans_summary_df

## 2 · Diversity metrics

In [ ]:
overview_df, failures_df = analyse_workload_set(
    WORKLOAD_NAMES,
    workload_root=WORKLOADS_DIR,
    lh=lh,
    catalog=TRINO_CATALOG,
    schema=TRINO_SCHEMA,
)
if not failures_df.empty:
    display(failures_df)

ANALYSIS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
overview_df.to_csv(WORKLOAD_METRICS_CSV, index=False)
print(f"wrote {WORKLOAD_METRICS_CSV}")
overview_df